# 🥈 Notebook 03: Silver Layer Transformations & Partitioning Experiments (`repartition` vs `coalesce`)

## 🎯 Objectives & "Why We Do This" Comparative Proofs
1. **Deduplication Proof**: Demonstrate why Window deduplication prevents revenue inflation.
2. **Quarantine Pattern Proof**: Demonstrate why routing corrupted rows to Quarantine is superior to dropping data.
3. **Partitioning Tuning Proof**: Comparative benchmark of **`repartition()` vs `coalesce()`** showing shuffle I/O cost and file count distribution.

---

In [0]:
# Databricks notebook source
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE_PATH = "/tmp/mini_project2/bronze"
SILVER_PATH = "/tmp/mini_project2/silver"

print(f"Reading Bronze Tables from: {BRONZE_PATH}")
print(f"Writing Silver Delta Tables to: {SILVER_PATH}")

### 💡 PROOF 1: Why Deduplicate using PySpark Window Functions (`row_number()`)?

In [0]:
df_bronze_orders = spark.read.format("delta").load(f"{BRONZE_PATH}/orders")
count_before_dedup = df_bronze_orders.count()

# Step A: Data Type Casting
df_typed_orders = df_bronze_orders \
    .withColumn("order_date", F.to_date(F.col("order_date"))) \
    .withColumn("total_amount", F.col("total_amount").cast("decimal(12,2)")) \
    .withColumn("_silver_processed_at", F.current_timestamp())

# Step B: Deduplication via Window function (keeping latest ingested record per order_id)
window_spec = Window.partitionBy("order_id").orderBy(F.col("_ingested_at").desc())
df_dedup_orders = df_typed_orders \
    .withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")

count_after_dedup = df_dedup_orders.count()
duplicates_removed = count_before_dedup - count_after_dedup

print("=== 📊 DEMONSTRATION: Deduplication Impact ===")
print(f"📦 Bronze Orders Row Count BEFORE Deduplication : {count_before_dedup}")
print(f"📦 Silver Orders Row Count AFTER Deduplication  : {count_after_dedup}")
print(f"🧹 Duplicate Rows Identified & Removed           : {duplicates_removed}")
print("💡 WHY WE DEDUPLICATE: Raw streaming/CSV pipelines often produce duplicate records due to retry policies. Deduplicating via Window functions ensures reporting metrics are not inflated!")

### 💡 PROOF 2: Why Quarantine Data instead of `df.dropna()`?

In [0]:
# Separate Valid Orders from Quarantined Orders
df_valid_orders = df_dedup_orders.filter(F.col("total_amount").isNotNull())
df_quarantine_orders = df_dedup_orders.filter(F.col("total_amount").isNull())

valid_count = df_valid_orders.count()
quarantine_count = df_quarantine_orders.count()

# Save Valid & Quarantine Tables
df_valid_orders.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}/orders")
df_quarantine_orders.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}/quarantine_orders")

print("=== 📊 DEMONSTRATION: Quarantine Pattern vs Dropping Rows ===")
print(f"✅ Valid Silver Orders Saved     : {valid_count} rows")
print(f"🚨 Quarantined Invalid Rows Saved: {quarantine_count} rows (missing total_amount)")
print("💡 WHY WE QUARANTINE: Silently dropping bad rows (`dropna()`) leads to unexplainable data loss during financial audits. Quarantining bad data routes corrupted records to an isolated table for data ops investigation and correction!")

### 💡 PROOF 3: Process Customers to Silver Layer

In [0]:
df_bronze_cust = spark.read.format("delta").load(f"{BRONZE_PATH}/customers")
df_silver_cust = df_bronze_cust \
    .withColumn("region", F.coalesce(F.trim(F.col("region")), F.lit("UNKNOWN"))) \
    .withColumn("tier", F.trim(F.col("tier"))) \
    .withColumn("signup_date", F.to_date(F.col("signup_date"))) \
    .withColumn("_silver_processed_at", F.current_timestamp())

df_silver_cust.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}/customers")
print(f"✅ Silver Customers saved: {df_silver_cust.count()} clean rows")

--- 
## 🧪 COMPARATIVE EXPERIMENT: `repartition()` vs `coalesce()`

### Why compare these two transformations?
- Developers often use `repartition()` and `coalesce()` interchangeably, leading to massive performance degradation or severe network bottlenecks.

In [0]:
# Setup Join Dataset
df_orders = spark.read.format("delta").load(f"{SILVER_PATH}/orders")
df_cust = spark.read.format("delta").load(f"{SILVER_PATH}/customers")
df_joined = df_orders.join(df_cust, "customer_id")
initial_partitions = df_joined.rdd.getNumPartitions()
print(f"Initial Partitions after join: {initial_partitions}")

In [0]:
# 🧪 EXPERIMENT A: repartition(10, 'region')
print("=== 📊 RUNNING EXPERIMENT A: repartition(10, 'region') ===")
t0 = time.time()

df_repart = df_joined.repartition(10, "region")
repart_output_path = f"{SILVER_PATH}/experiment_repartition"
df_repart.write.format("delta").mode("overwrite").save(repart_output_path)

t1 = time.time()
repart_time = round(t1 - t0, 3)
repart_files = [f for f in dbutils.fs.ls(repart_output_path) if f.path.endswith(".parquet")]

print(f"⏱️ Repartition Execution Time: {repart_time} seconds")
print(f"📂 Partitions: {df_repart.rdd.getNumPartitions()} | Parquet Files Written: {len(repart_files)}")
print("🔍 Physical Plan (Notice 'Exchange hashpartitioning' - FULL SHUFFLE COST):")
df_repart.explain(True)

In [0]:
# 🧪 EXPERIMENT B: coalesce(1)
print("=== 📊 RUNNING EXPERIMENT B: coalesce(1) ===")
t0 = time.time()

# Filter to simulate result set, then coalesce
df_filtered = df_joined.filter(F.col("region") == "APAC")
df_coalesce = df_filtered.coalesce(1)
coalesce_output_path = f"{SILVER_PATH}/experiment_coalesce"
df_coalesce.write.format("delta").mode("overwrite").save(coalesce_output_path)

t1 = time.time()
coalesce_time = round(t1 - t0, 3)
coalesce_files = [f for f in dbutils.fs.ls(coalesce_output_path) if f.path.endswith(".parquet")]

print(f"⏱️ Coalesce Execution Time: {coalesce_time} seconds")
print(f"📂 Partitions: {df_coalesce.rdd.getNumPartitions()} | Parquet Files Written: {len(coalesce_files)}")
print("🔍 Physical Plan (Notice ABSENCE of 'Exchange' shuffle operator):")
df_coalesce.explain(True)

## 💡 PROOF SUMMARY: Why choose `repartition()` vs `coalesce()`?

```
=== DECISION SUMMARY ===
1. USE repartition(N, col) WHEN:
   - You need to increase partition count for higher parallelism.
   - Data is heavily skewed across worker nodes and needs re-balancing via hash shuffle.
   - Preparing dataset for heavy join or aggregation operations.

2. USE coalesce(N) WHEN:
   - You are reducing partition count (e.g. after a filter) to avoid the Small File Problem.
   - You want to eliminate the expensive network shuffle penalty entirely!
```